# Explore NYC Taxi Bronze table

In [2]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "warehouse").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root."
    )


def find_java_home() -> Path:
    candidates = [
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    for candidate in candidates:
        if (candidate / "bin" / "java").exists():
            return candidate
    raise FileNotFoundError(
        "No JDK 17/21 found. Install one with: brew install openjdk@21"
    )


PROJECT_ROOT = find_project_root()
os.environ["JAVA_HOME"] = str(find_java_home())
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

CATALOG_DB = PROJECT_ROOT / "data" / "catalog.db"
WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"

SPARK_PACKAGES = ",".join(
    [
        "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0",
        "org.xerial:sqlite-jdbc:3.49.1.0",
    ]
)

spark = (
    SparkSession.builder
    .appName("nyc-taxi-lakehouse")
    .master("local[*]")
    .config("spark.jars.packages", SPARK_PACKAGES)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    .config("spark.sql.catalog.local.uri", f"jdbc:sqlite:{CATALOG_DB}")
    .config("spark.sql.catalog.local.warehouse", WAREHOUSE_DIR.as_uri())
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

BRONZE_TABLE = "local.bronze.yellow_trips"

print("Project root:", PROJECT_ROOT)
print("Java home:", os.environ["JAVA_HOME"])


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/12 02:30:38 WARN Utils: Your hostname, admins-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.125 instead (on interface en0)
26/07/12 02:30:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ittipat/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ittipat/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
org.xerial#sqlite-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b85c1f0f-010d-489f-a97f-ef3574d1066f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
	found org.xerial#sqlite-jdbc;3.49.1.0 in central
:: resolution report :: 

Project root: /Users/ittipat/Desktop/nyc-taxi-lakehouse-demo
Java home: /opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


In [3]:
bronze_df = spark.table(BRONZE_TABLE)

bronze_df.printSchema()
row_count = bronze_df.count()
print(f"Bronze rows: {row_count:,}")

26/07/12 02:31:41 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- _bronze_source_file: string (nullable = false)
 |-- _bronze_ingested_at

In [4]:
bronze_df.show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------------------+--------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|_bronze_source_file            |_bronze_ingested_at       |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------

In [5]:
(
    bronze_df
    .groupBy("_bronze_source_file")
    .count()
    .orderBy("_bronze_source_file")
    .show(truncate=False)
)

+-------------------------------+-------+
|_bronze_source_file            |count  |
+-------------------------------+-------+
|yellow_tripdata_2026-01.parquet|3724889|
|yellow_tripdata_2026-02.parquet|3399866|
|yellow_tripdata_2026-03.parquet|3952451|
+-------------------------------+-------+



In [6]:
numeric_columns = [
    field.name
    for field in bronze_df.schema.fields
    if field.dataType.typeName() in {
        "byte", "short", "integer", "long", "float", "double", "decimal"
    }
]

bronze_df.select(*numeric_columns).summary().show(truncate=False)

26/07/12 02:32:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|VendorID          |passenger_count   |trip_distance    |RatecodeID       |PULocationID     |DOLocationID      |payment_type      |fare_amount       |extra             |mta_tax            |tip_amount       |tolls_amount      |improvement_surcharge|total_amount      |congestion_surcharge|Airport_fee        |cbd_congestion_fee|
+-------+------------------+------------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------

In [19]:
null_counts = bronze_df.agg(
    *[
        F.sum(F.col(column).isNull().cast("long")).alias(column)
        for column in bronze_df.columns
    ]
).first().asDict()

null_summary = spark.createDataFrame(
    [
        (
            column,
            int(null_counts[column]),
            round(int(null_counts[column]) / row_count * 100, 2) if row_count else 0.0,
        )
        for column in bronze_df.columns
    ],
    ["column", "null_count", "null_percentage"],
)

null_summary.orderBy(
    F.desc("null_percentage"),
    F.desc("null_count"),
).show(len(bronze_df.columns), truncate=False)

+---------------------+----------+---------------+
|column               |null_count|null_percentage|
+---------------------+----------+---------------+
|congestion_surcharge |3057123   |27.6           |
|RatecodeID           |3057123   |27.6           |
|Airport_fee          |3057123   |27.6           |
|store_and_fwd_flag   |3057123   |27.6           |
|passenger_count      |3057123   |27.6           |
|PULocationID         |0         |0.0            |
|tolls_amount         |0         |0.0            |
|DOLocationID         |0         |0.0            |
|cbd_congestion_fee   |0         |0.0            |
|improvement_surcharge|0         |0.0            |
|extra                |0         |0.0            |
|VendorID             |0         |0.0            |
|payment_type         |0         |0.0            |
|_bronze_source_file  |0         |0.0            |
|fare_amount          |0         |0.0            |
|tpep_pickup_datetime |0         |0.0            |
|_bronze_ingested_at  |0       

In [20]:
(
    bronze_df
    .filter(
        F.col("passenger_count").isNull()
        & F.col("RatecodeID").isNull()
        & F.col("congestion_surcharge").isNull()
        & F.col("store_and_fwd_flag").isNull()
        & F.col("Airport_fee").isNull()
    )
    .count()
)

3057123

In [22]:
(
    bronze_df
    .filter(
        F.col("passenger_count").isNull()
        & F.col("RatecodeID").isNull()
        & F.col("congestion_surcharge").isNull()
        & F.col("store_and_fwd_flag").isNull()
        & F.col("Airport_fee").isNull()
    )
    .groupBy("_bronze_source_file")
    .count()
    .show(truncate=False)
)

+-------------------------------+-------+
|_bronze_source_file            |count  |
+-------------------------------+-------+
|yellow_tripdata_2026-03.parquet|945748 |
|yellow_tripdata_2026-02.parquet|1023317|
|yellow_tripdata_2026-01.parquet|1088058|
+-------------------------------+-------+

